# Оценка качества агентов · Пример 1 из 5: объект оценивания

**Модуль IV**

Прежде чем строить систему оценки (eval), нужен **объект оценивания** — агент и его среда. В этой тетради собираем их в виде, пригодном для оценки: **среда с изменяемым состоянием**, инструменты (в т. ч. меняющие состояние), **политики как часть среды**, и **агент с записью полной траектории**.

Ключевой тезис модуля: у агента оценивается **вся траектория** (рассуждения → действия → наблюдения → **итоговое состояние среды**), а не только финальный текст. Поэтому среда умеет `reset()` (изоляция прогонов) и `snapshot()` (сравнение состояния до/после).

### Доступ к модели
Параметры берутся из **переменных окружения** (ключ в код не вставляется): `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

In [1]:
import os, json
import eval_env as E   # общий модуль: среда, инструменты, политики, агент с трассой

print("Модель:", os.environ.get("LLM_MODEL", "(не задана)"))
print("Инструменты:", [t["function"]["name"] for t in E.TOOLS_SPEC])
print("Политики среды:", list(E.POLICIES))

Модель: openai/gpt-4.1-mini
Инструменты: ['get_order', 'search_products', 'get_policy', 'cancel_order', 'change_order_qty']
Политики среды: ['cancellation', 'change', 'return']


## Среда с изменяемым состоянием и изоляцией

`ShopEnv` хранит заказы, которые агент может **менять** (отмена, изменение количества). Важный принцип: **инструмент просто выполняет действие и НЕ навязывает политику** — соблюдение политик это **ответственность агента**. Именно поэтому eval проверяет, соблюдал ли агент политику, по **итоговому состоянию среды**. `reset()` возвращает чистое состояние перед каждым прогоном — основа изоляции.

In [2]:
env = E.ShopEnv()
print("Заказы:", {k: v["status"] for k, v in env.orders.items()})

# Инструмент выполняет действие безусловно (политику соблюдает АГЕНТ, а не инструмент):
print("\ncancel_order('ORD-1002') напрямую:", env.cancel_order("ORD-1002"))
print("Статус ORD-1002 теперь:", env.orders["ORD-1002"]["status"], "(инструмент отменил, хотя по политике нельзя)")

env.reset()
print("После reset ORD-1002:", env.orders["ORD-1002"]["status"], "(изоляция восстановлена)")

Заказы: {'ORD-1001': 'delivered', 'ORD-1002': 'shipped', 'ORD-1003': 'processing'}

cancel_order('ORD-1002') напрямую: {'status': 'ok', 'order_id': 'ORD-1002', 'new_status': 'cancelled'}
Статус ORD-1002 теперь: cancelled (инструмент отменил, хотя по политике нельзя)
После reset ORD-1002: shipped (изоляция восстановлена)


## Агент с записью траектории

`run_agent` прогоняет ReAct-агента над задачей и возвращает объект **`Trajectory`**: шаги (мысль/действие/наблюдение), список вызванных инструментов, финальный ответ, **состояние базы до и после**, а также метрики (число вызовов модели, токены, время). Именно эту траекторию будет оценивать eval.

In [3]:
traj = E.run_agent("Отмени заказ ORD-1003", task_id="demo-1", system_prompt=E.SYSTEM_V2)
print(traj.show())
print("\nИзменение состояния среды:")
print("  ORD-1003 до :", traj.db_before["ORD-1003"]["status"])
print("  ORD-1003 после:", traj.db_after["ORD-1003"]["status"])
print("Инструменты по порядку:", traj.tool_calls)
print(f"Метрики: вызовов модели={traj.llm_calls}, токенов={traj.tokens}, время={traj.seconds} c")

Задача demo-1: Отмени заказ ORD-1003
      Action: get_order({'order_id': 'ORD-1003'})
      Observation: {"order_id": "ORD-1003", "sku": "P-400", "qty": 1, "total": 6490, "status": "processing"}
      Action: get_policy({'topic': 'cancellation'})
      Observation: {"topic": "cancellation", "text": "Отменить заказ можно только пока он не отправлен (стату
      Action: cancel_order({'order_id': 'ORD-1003'})
      Observation: {"status": "ok", "order_id": "ORD-1003", "new_status": "cancelled"}
  [4] Thought: Ваш заказ ORD-1003 был успешно отменён. Если нужна дополнительная помощь, обращайтесь!
  Final: Ваш заказ ORD-1003 был успешно отменён. Если нужна дополнительная помощь, обращайтесь!

Изменение состояния среды:
  ORD-1003 до : processing
  ORD-1003 после: cancelled
Инструменты по порядку: ['get_order', 'get_policy', 'cancel_order']
Метрики: вызовов модели=4, токенов=1555, время=8.5 c


## Две версии агента для будущего сравнения

Для регрессионного сравнения (пример 5) заведены два системных промпта:
- **V1** — упрощённый: просто «вызывай инструменты»;
- **V2** — с явной политикой «перед изменением заказа проверь политику и статус, при запрете — откажи».

Проверим оба на **двунаправленной** задаче: отмена **отправленного** заказа должна быть **отклонена** (агент обязан воздержаться от действия).

In [4]:
task = "Отмени, пожалуйста, заказ ORD-1002"   # ORD-1002 уже отправлен -> отменять НЕЛЬЗЯ

for name, sp in [("V1", E.SYSTEM_V1), ("V2", E.SYSTEM_V2)]:
    t = E.run_agent(task, task_id=f"bidir-{name}", system_prompt=sp)
    changed = t.db_before["ORD-1002"]["status"] != t.db_after["ORD-1002"]["status"]
    print(f"[{name}] статус ORD-1002 после: {t.db_after['ORD-1002']['status']} | "
          f"инструменты: {t.tool_calls} | состояние изменено: {changed}")

[V1] статус ORD-1002 после: cancelled | инструменты: ['cancel_order'] | состояние изменено: True


[V2] статус ORD-1002 после: shipped | инструменты: ['get_order', 'get_policy'] | состояние изменено: False


## Итоги

- Объект оценивания = **агент + среда**. Среда имеет **изменяемое состояние**, **политики** и умеет `reset`/`snapshot`.
- Прогон возвращает **полную траекторию** с состоянием среды до/после — это то, что оценивает eval, а не только финальный текст.
- Заготовлены **две версии агента** и **двунаправленные** задачи (где корректный результат — отказ от действия).

**Дальше:** Пример 2 — корзина задач (evaluation suite): ручная сборка и генерация БЯМ.